# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

# import importlib.util
# import sys

# Check if 'pysqlite3' is available before importing
# if importlib.util.find_spec("pysqlite3") is not None:
#     import pysqlite3
#     sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
from typing import List, Dict, Optional, Union
from pydantic import BaseModel
import os
from datetime import datetime
import json
import chromadb
from tavily import TavilyClient
from dotenv import load_dotenv

from lib.agents import Agent, AgentState
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, BaseMessage
from lib.tooling import tool, Tool
from lib.parsers import JsonOutputParser
from lib.memory import ShortTermMemory
from lib.state_machine import StateMachine, EntryPoint, Step, Termination, Run

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
@tool
def retrieve_game(query: str) -> List[Dict[str, str]]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    results = collection.query(
        query_texts=[query],
        n_results=5,
    )
    formatted_results = []
    for i in range(len(results["ids"][0])):
        result = {
            "Platform": results["metadatas"][0][i]["Platform"],
            "Name": results["metadatas"][0][i]["Name"],
            "YearOfRelease": results["metadatas"][0][i]["YearOfRelease"],
            "Description": results["documents"][0][i],
        }
        formatted_results.append(result)
    return formatted_results  # return list of results

In [5]:
# Test retrieve_game tool
documents = retrieve_game("When Pokémon Gold and Silver was released?")
print("Retrieved Documents:")
print(documents)

Retrieved Documents:
[{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': '[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': '[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': "[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}, {'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Description': '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic plat

#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> str:
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini")
    prompt = f"""
    Your task is to evaluate if the documents are enough to respond the query.
    Question: {question}
    Retrieved Documents: {retrieved_docs}
    
    Please provide a detailed explanation, so it's possible to take an action to accept it or not.
    """
    evaluation = llm.invoke(prompt, EvaluationReport)
    parser = JsonOutputParser()
    return parser.parse(evaluation)

In [7]:
# Test evaluate_retrieval tool
documents = retrieve_game("When Pokémon Gold and Silver was released?")
evaluation_result = evaluate_retrieval(
    question="When Pokémon Gold and Silver was released?",
    retrieved_docs=documents
)
print("Evaluation Result:")
print(evaluation_result)

Evaluation Result:
{'useful': True, 'description': 'The retrieved documents contain sufficient information to answer the query regarding the release date of Pokémon Gold and Silver. Specifically, one of the documents explicitly states that Pokémon Gold and Silver was released in 1999. This directly addresses the question asked. \n\nWhile there are other documents included that pertain to different games and platforms, they do not detract from the relevance of the document that contains the necessary information. Therefore, the documents are adequate to respond to the query.'}


#### Game Web Search Tool

In [8]:
@tool
def game_web_search(question: str) -> str:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry. 
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    
    # Perform the search
    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }
    
    return formatted_results

In [9]:
# Test game_web_search tool
search_results = game_web_search("When was Pokémon Gold and Silver released?")
print("Search Results:")
print(search_results)

Search Results:
{'answer': 'Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000.', 'results': [{'url': 'https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver', 'title': 'Pokémon Gold and Silver - Wikipedia', 'content': 'In September 1999, Nintendo announced that Gold and Silver would be released in North America in September 2000. In May 2000, Nintendo announced the official North American release date of Gold and Silver would instead be October 16 of that year. The release date was later changed to October 15. In North America, Nintendo started accepting pre-orders for the games in August; a CD-ROM was available as a pre-order bonus that included clips and music from Pokémon the Movie 2000, screenshots from Pokémon Gold and Silver, a Pokémon-themed desktop wallpaper, an offer for a Nintendo Power Player\'s Guide, and Pokémon-related trivia. The games had record pre-order sales — approximately 600,000 copies of the games we

### Agent

In [10]:
class MemoryAgent:
    def __init__(self, 
                 model_name: str,
                 instructions: str, 
                 tools: List[Tool] = None,
                 temperature: float = 0.7):
        """
        Initialize a MemoryAgent instance
        
        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature
        
        self.memory = ShortTermMemory()
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        """Step logic: Prepare messages for LLM consumption"""
        messages = state.get("messages", [])
        
        # If no messages exist, start with system message
        if not messages:
            messages = [SystemMessage(content=state["instructions"])]
            
        # Add the new user message
        messages.append(UserMessage(content=state["user_query"]))
        
        return {
            "messages": messages,
            "session_id": state["session_id"]
        }

    def _llm_step(self, state: AgentState) -> AgentState:
        """Step logic: Process the current state through the LLM"""
        # Initialize LLM
        llm = LLM(
            model=self.model_name,
            temperature=self.temperature,
            tools=self.tools
        )

        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls if response.tool_calls else None

        # Create AI message with content and tool calls
        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)
        
        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls,
            "session_id": state["session_id"]
        }

    def _tool_step(self, state: AgentState) -> AgentState:
        """Step logic: Execute any pending tool calls"""
        tool_calls = state["current_tool_calls"] or []
        tool_messages = []
        
        for call in tool_calls:
            # Access tool call data correctly
            function_name = call.function.name
            function_args = json.loads(call.function.arguments)
            tool_call_id = call.id
            # Find the matching tool
            tool = next((t for t in self.tools if t.name == function_name), None)
            if tool:
                result = tool(**function_args)
                tool_message = ToolMessage(
                    content=json.dumps(result), 
                    tool_call_id=tool_call_id, 
                    name=function_name, 
                )
                tool_messages.append(tool_message)
        
        # Clear tool calls and add results to messages
        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"]
        }

    def _create_state_machine(self) -> StateMachine[AgentState]:
        """Create the internal state machine for the agent"""
        machine = StateMachine[AgentState](AgentState)
        
        # Create steps
        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[AgentState]("llm_processor", self._llm_step)
        tool_executor = Step[AgentState]("tool_executor", self._tool_step)
        termination = Termination[AgentState]()
        
        machine.add_steps([entry, message_prep, llm_processor, tool_executor, termination])
        
        # Add transitions
        machine.connect(entry, message_prep)
        machine.connect(message_prep, llm_processor)
        
        # Transition based on whether there are tool calls
        def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
            """Transition logic: Check if there are tool calls"""
            if state.get("current_tool_calls"):
                return tool_executor
            return termination
        
        machine.connect(llm_processor, [tool_executor, termination], check_tool_calls)
        machine.connect(tool_executor, llm_processor)  # Go back to llm after tool execution
        
        return machine

    def invoke(self, query: str, session_id: Optional[str] = None) -> Run:
        """
        Run the agent on a query
        
        Args:
            query: The user's query to process
            session_id: Optional session identifier (uses "default" if None)
            
        Returns:
            The final run object after processing
        """
        session_id = session_id or "default"

        self.memory.create_session(session_id)

        # Get previous messages from last run if available
        previous_messages = []
        last_run: Run = self.memory.get_last_object(session_id)
        if last_run:
            last_state = last_run.get_final_state()
            if last_state:
                previous_messages = last_state["messages"]

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": previous_messages,
            "current_tool_calls": None,
            "session_id": session_id,
        }

        run_object = self.workflow.run(initial_state)
        
        self.memory.add(run_object, session_id)
        
        return run_object

    def get_session_runs(self, session_id: Optional[str] = None) -> List[Run]:
        """Get all Run objects for a session
        
        Args:
            session_id: Optional session ID (uses "default" if None)
            
        Returns:
            List of Run objects in the session
        """
        session_id = session_id or "default"
        return self.memory.get_all_objects(session_id)

    def reset_session(self, session_id: Optional[str] = None):
        """Reset memory for a specific session
        
        Args:
            session_id: Optional session to reset (uses "default" if None)
        """
        session_id = session_id or "default"
        self.memory.reset(session_id)

In [11]:
udaplay_agent = MemoryAgent(
    model_name="gpt-4o-mini",
    instructions="""
    You are UdaPlay, an expert assistant in the gaming industry.
    Your task is to help users find accurate and relevant information about video games.
    Start by retrieving relevant documents from the vector database using the 'retrieve_game' tool.
    Next, evaluate the usefulness of the retrieved documents with the 'evaluate_retrieval' tool.
    Only if the documents are not sufficient, use the 'game_web_search' tool to find additional information from the web.
    Finally, provide a comprehensive answer to the user's question based on the gathered information.
    """,
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)

In [12]:
run = udaplay_agent.invoke("When Pokémon Gold and Silver was released?", session_id="gaming_session_1")
print("Agent Response:")
run.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent Response:


[SystemMessage(role='system', content="\n    You are UdaPlay, an expert assistant in the gaming industry.\n    Your task is to help users find accurate and relevant information about video games.\n    Start by retrieving relevant documents from the vector database using the 'retrieve_game' tool.\n    Next, evaluate the usefulness of the retrieved documents with the 'evaluate_retrieval' tool.\n    Only if the documents are not sufficient, use the 'game_web_search' tool to find additional information from the web.\n    Finally, provide a comprehensive answer to the user's question based on the gathered information.\n    "),
 UserMessage(role='user', content='When Pokémon Gold and Silver was released?'),
 AIMessage(role='assistant', content=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XIYxkTXkqnP6vdo2IvHGvLxw', function=Function(arguments='{"query":"Pokémon Gold and Silver release date"}', name='retrieve_game'), type='function')], token_usage=None),
 ToolMessage(role='

In [13]:
run = udaplay_agent.invoke("Which one was the first 3D platformer Mario game?", session_id="gaming_session_2")
print("Agent Response:")
run.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent Response:


[SystemMessage(role='system', content="\n    You are UdaPlay, an expert assistant in the gaming industry.\n    Your task is to help users find accurate and relevant information about video games.\n    Start by retrieving relevant documents from the vector database using the 'retrieve_game' tool.\n    Next, evaluate the usefulness of the retrieved documents with the 'evaluate_retrieval' tool.\n    Only if the documents are not sufficient, use the 'game_web_search' tool to find additional information from the web.\n    Finally, provide a comprehensive answer to the user's question based on the gathered information.\n    "),
 UserMessage(role='user', content='Which one was the first 3D platformer Mario game?'),
 AIMessage(role='assistant', content=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Q5CnEgaic2VSf5AQ74bh1cwr', function=Function(arguments='{"query":"first 3D platformer Mario game"}', name='retrieve_game'), type='function')], token_usage=None),
 ToolMessage(role=

In [14]:
run = udaplay_agent.invoke("Was Mortal Kombat X realeased for Playstation 5?", session_id="gaming_session_3")
print("Agent Response:")
run.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Agent Response:


[SystemMessage(role='system', content="\n    You are UdaPlay, an expert assistant in the gaming industry.\n    Your task is to help users find accurate and relevant information about video games.\n    Start by retrieving relevant documents from the vector database using the 'retrieve_game' tool.\n    Next, evaluate the usefulness of the retrieved documents with the 'evaluate_retrieval' tool.\n    Only if the documents are not sufficient, use the 'game_web_search' tool to find additional information from the web.\n    Finally, provide a comprehensive answer to the user's question based on the gathered information.\n    "),
 UserMessage(role='user', content='Was Mortal Kombat X realeased for Playstation 5?'),
 AIMessage(role='assistant', content=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_XRdi80uEIGyj2cvs9by3Diyu', function=Function(arguments='{"query":"Mortal Kombat X release for Playstation 5"}', name='retrieve_game'), type='function')], token_usage=None),
 ToolMes

### (Optional) Advanced

In [15]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes